# 学ぬい分類用転移学習

MobileNetV3 Large をベースに転移学習するこどで、学ぬいを分類するモデルをつくる。

方針:
- `data/processed/` 以下のディレクトリ構造（`{category_name}/*.jpg`）をそのまま `ImageFolder` として利用する
- train/val は層化(stratify)分割する
- train用とval用でtransform（データ拡張の有無）を分ける
- まずはバックボーンを凍結し、分類層のみ学習する
- 保存時は重みと一緒に `class_to_idx` も保存する（推論側でラベル対応が必要になるため）

In [2]:
import random
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import MobileNet_V3_Large_Weights, mobilenet_v3_large

## 設定

In [3]:
# 値は適宜調整
CONFIG = {
    "data_dir": Path("../data/processed"),
    "model_dir": Path("../models"),
    "seed": 42,
    "val_ratio": 0.2,
    "batch_size": 16,
    "num_epochs": 15,
    "lr": 1e-3,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
print(CONFIG)

{'data_dir': PosixPath('../data/processed'), 'model_dir': PosixPath('../models'), 'seed': 42, 'val_ratio': 0.2, 'batch_size': 16, 'num_epochs': 15, 'lr': 0.001, 'device': 'cpu'}


## データ拡張 / 前処理

trainのみ拡張をかける。valは拡張なし（リサイズ・正規化のみ）。

In [4]:
# 事前学習のパラメータを使用する
# 転移学習では、画像のRGB値（[0,1)）の分布（平均と偏差）を事前学習のデータセットと同じものとする必要があるため
weights = MobileNet_V3_Large_Weights.DEFAULT
preset = weights.transforms()
mean, std = preset.mean, preset.std

train_transform = transforms.Compose([
    # ぬいの位置を画像内である程度ランダムに動かすことで、ユーザーがぬいをカメラの端で捉えたときにも適応できるようにする、という理解
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    # Horizontal Flipはスマホのインカメラ使用時などを想定して必要そう
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# ↑みたいな幾何処理がいらない場合はpresetをそのまま使うと楽
# preprocessで画像サイズをそろえているのでリサイズはここでしなくていいのも貢献してる。
val_transform = weights.transforms()

## データセット / 層化分割

`ImageFolder` はインスタンスごとに1つのtransformしか持てないため、train用・val用で別インスタンスを作り、
同じインデックスを`Subset`で使い分ける。

In [5]:
from collections import defaultdict

dataset_train_view = ImageFolder(CONFIG["data_dir"], transform=train_transform)
dataset_val_view = ImageFolder(CONFIG["data_dir"], transform=val_transform)

print("classes:", dataset_train_view.classes)
print("class_to_idx:", dataset_train_view.class_to_idx)


def stratified_split(dataset, val_ratio: float, seed: int) -> tuple[list[int], list[int]]:
    """クラスごとにシャッフルしてval_ratio分をvalに回し、train/valのインデックスを返す"""
    rng = random.Random(seed)
    
    class_indices = defaultdict(list)
    for idx, label in enumerate(dataset.targets):
        class_indices[label].append(idx)
    
    train_indices = []
    val_indices = []
    
    for label_indices in class_indices.values():
        rng.shuffle(label_indices)
        val_count = int(len(label_indices) * val_ratio)
        val_indices += label_indices[:val_count]
        train_indices += label_indices[val_count:]

    return (train_indices, val_indices)

train_indices, val_indices = stratified_split(
    dataset_train_view, CONFIG["val_ratio"], CONFIG["seed"]
)

train_dataset = Subset(dataset_train_view, train_indices)
val_dataset = Subset(dataset_val_view, val_indices)

print(f"train: {len(train_dataset)} / val: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

classes: ['kotone', 'saki']
class_to_idx: {'kotone': 0, 'saki': 1}
train: 154 / val: 37


## モデル構築（転移学習）

Classifierの最終層以外（バックボーン（畳み込み層の積み重ね）とclassifier内の最初のレイヤ）を凍結し、分類層の最終Linearだけ `num_classes` に差し替える。新たに学習するデータ量は元のモデルの学習データより非常に小さいので、凍結をしないと過学習に陥る。転移学習の定跡。

In [6]:
def build_model(num_classes: int) -> nn.Module:
    model = mobilenet_v3_large(weights=weights)

    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier[0].parameters():
        param.requires_grad = False

    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    
    return model


model = build_model(num_classes=len(dataset_train_view.classes)).to(CONFIG["device"])

## 学習ループ

In [9]:
criterion = nn.CrossEntropyLoss()

# 凍結していないパラメータのみoptimizerに渡す
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["lr"],
)


def train_one_epoch(model, loader, criterion, optimizer, device) -> float:
    """1エポック分学習し、平均lossを返す"""
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device) -> tuple[float, float]:
    """評価して (平均loss, accuracy) を返す"""
    model.eval()
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
        
    return (total_loss / len(loader.dataset), correct / len(loader.dataset))

In [ ]:
best_val_acc = 0.0
CONFIG["model_dir"].mkdir(parents=True, exist_ok=True)

for epoch in range(CONFIG["num_epochs"]):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, CONFIG["device"])
    val_loss, val_acc = evaluate(model, val_loader, criterion, CONFIG["device"])

    print(
        f"epoch {epoch + 1}/{CONFIG['num_epochs']} "
        f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )
    
    # ベスト更新時にチェックポイントを保存
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "class_to_idx": dataset_train_view.class_to_idx,
            },
            CONFIG["model_dir"] / "mobilenet_v3_large_nui.pth",
        )


epoch 1/15 train_loss=0.4013 val_loss=0.4426 val_acc=0.8108
epoch 2/15 train_loss=0.3409 val_loss=0.4145 val_acc=0.8378
epoch 3/15 train_loss=0.3584 val_loss=0.3942 val_acc=0.8378
epoch 4/15 train_loss=0.2959 val_loss=0.3769 val_acc=0.7838
epoch 5/15 train_loss=0.2833 val_loss=0.3687 val_acc=0.8108
epoch 6/15 train_loss=0.2590 val_loss=0.3709 val_acc=0.7838
epoch 7/15 train_loss=0.2774 val_loss=0.3677 val_acc=0.7838
epoch 8/15 train_loss=0.2869 val_loss=0.3609 val_acc=0.7838
epoch 9/15 train_loss=0.2618 val_loss=0.3507 val_acc=0.8378
epoch 10/15 train_loss=0.2198 val_loss=0.3497 val_acc=0.8378
epoch 11/15 train_loss=0.2118 val_loss=0.3507 val_acc=0.8378
epoch 12/15 train_loss=0.2185 val_loss=0.3506 val_acc=0.8378
epoch 13/15 train_loss=0.1915 val_loss=0.3486 val_acc=0.8919
epoch 14/15 train_loss=0.1986 val_loss=0.3515 val_acc=0.8919
epoch 15/15 train_loss=0.2254 val_loss=0.3492 val_acc=0.8649
